# Edinburgh Airbnb Exploratory Data Analysis

## Section 04 — EDA

This notebook explores the Edinburgh Airbnb market using the processed DuckDB
warehouse created in Section 03.

## Objectives

- Analyse price distributions by neighbourhood and room type
- Examine host supply concentration
- Explore review score patterns
- Investigate occupancy proxy and revenue estimates
- Translate findings into business interpretations

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

database_path = project_root / "database" / "airbnb.duckdb"
figures_dir = project_root / "reports" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

connection = duckdb.connect(str(database_path))

eda_query = """
SELECT
    f.listing_id,
    f.host_id,
    n.neighbourhood,
    f.price,
    f.room_type,
    f.property_type,
    f.number_of_reviews,
    f.review_scores_rating,
    f.availability_365,
    f.occupancy_proxy,
    f.estimated_revenue_proxy,
    f.price_per_bedroom,
    h.host_is_superhost,
    h.host_listings_count
FROM dw.fact_listing_performance f
LEFT JOIN dw.dim_neighbourhood n
    ON f.neighbourhood_key = n.neighbourhood_key
LEFT JOIN dw.dim_host h
    ON f.host_id = h.host_id
WHERE f.is_valid_price = TRUE
  AND f.is_valid_location = TRUE
"""

listings = connection.execute(eda_query).fetchdf()

print(f"Project root: {project_root}")
print(f"Rows used for EDA: {len(listings):,}")
listings.head()

## 1. Price Distribution by Neighbourhood

This chart shows which neighbourhoods command the highest median nightly prices.

In [ ]:
top_neighbourhoods = (
    listings.groupby("neighbourhood", as_index=False)
    .agg(median_price=("price", "median"), listing_count=("listing_id", "count"))
    .query("listing_count >= 10")
    .sort_values("median_price", ascending=False)
    .head(15)
)

plt.figure(figsize=(12, 6))
sns.barplot(
    data=top_neighbourhoods,
    x="median_price",
    y="neighbourhood",
    hue="neighbourhood",
    palette="viridis",
    legend=False,
)
plt.title("Top 15 Edinburgh Neighbourhoods by Median Listing Price")
plt.xlabel("Median nightly price (£)")
plt.ylabel("Neighbourhood")
plt.tight_layout()
plt.savefig(figures_dir / "01_median_price_by_neighbourhood.png", dpi=150)
plt.show()

top_neighbourhoods

### Business interpretation

Neighbourhoods with the highest median prices are likely close to tourist demand
drivers such as the city centre, festival corridors, or premium residential areas.
For investors, these areas may offer higher revenue potential but also stronger
competition. For new hosts, entering a high-price neighbourhood requires stronger
differentiation because guests already expect premium pricing.

## 2. Price Distribution by Room Type

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=listings,
    x="room_type",
    y="price",
    hue="room_type",
    palette="Set2",
    legend=False,
)
plt.title("Listing Price Distribution by Room Type")
plt.xlabel("Room type")
plt.ylabel("Nightly price (£)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(figures_dir / "02_price_by_room_type.png", dpi=150)
plt.show()

listings.groupby("room_type")["price"].agg(["count", "median", "mean"]).round(2)

### Business interpretation

Entire homes/apartments typically sit above private and shared rooms in both median
and upper-range prices. This supports a premium positioning strategy for whole-property
listings. Private rooms may appeal to budget travellers and solo visitors, while entire
homes are better suited to families and longer-stay guests willing to pay more for privacy.

## 3. Host Supply Concentration

How many listings does each host control?

In [ ]:
host_portfolio = (
    listings.groupby("host_id", as_index=False)
    .agg(listing_count=("listing_id", "count"))
)

plt.figure(figsize=(10, 6))
sns.histplot(
    host_portfolio["listing_count"],
    bins=30,
    kde=False,
)
plt.title("Distribution of Listings per Host")
plt.xlabel("Listings per host")
plt.ylabel("Number of hosts")
plt.tight_layout()
plt.savefig(figures_dir / "03_host_portfolio_distribution.png", dpi=150)
plt.show()

single_listing_hosts = (host_portfolio["listing_count"] == 1).mean() * 100
multi_listing_hosts = (host_portfolio["listing_count"] > 1).mean() * 100

print(f"Hosts with one listing: {single_listing_hosts:.1f}%")
print(f"Hosts with multiple listings: {multi_listing_hosts:.1f}%")

### Business interpretation

If most hosts operate only one listing, the market is still dominated by casual or
independent operators rather than large commercial property managers. However, hosts
with multiple listings can have outsized influence on neighbourhood supply. A market
intelligence platform should monitor multi-listing hosts separately because they may
behave more like professional operators with different pricing and occupancy strategies.

## 4. Review Score Distribution

In [ ]:
reviewed_listings = listings.dropna(subset=["review_scores_rating"])

plt.figure(figsize=(10, 6))
sns.histplot(
    reviewed_listings["review_scores_rating"],
    bins=20,
    kde=True,
)
plt.title("Distribution of Overall Review Scores")
plt.xlabel("Review score")
plt.ylabel("Number of listings")
plt.tight_layout()
plt.savefig(figures_dir / "04_review_score_distribution.png", dpi=150)
plt.show()

reviewed_listings["review_scores_rating"].describe().round(2)

### Business interpretation

A right-skewed distribution concentrated above 4.5 may indicate rating inflation or
guest reluctance to leave very low public scores. For Airbnb operators, maintaining a
high rating is necessary but no longer sufficient for differentiation because many
listings cluster at the top end. Competitive advantage may depend more on review volume,
location, and amenity fit than on small differences between 4.8 and 4.9.

## 5. Occupancy Proxy by Room Type

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=listings,
    x="room_type",
    y="occupancy_proxy",
    hue="room_type",
    estimator="mean",
    errorbar=None,
    palette="crest",
    legend=False,
)
plt.title("Average Occupancy Proxy by Room Type")
plt.xlabel("Room type")
plt.ylabel("Occupancy proxy")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(figures_dir / "05_occupancy_proxy_by_room_type.png", dpi=150)
plt.show()

listings.groupby("room_type")["occupancy_proxy"].mean().round(4)

### Business interpretation

Occupancy proxy differences across room types suggest that some listing categories
experience stronger calendar pressure than others. Entire homes may show different
availability patterns from private rooms depending on seasonality and trip purpose.
This metric should be treated as a demand signal, not confirmed occupancy, but it
still helps identify which segments appear more heavily utilised.

## 6. Price vs Review Score

In [ ]:
scatter_data = listings.dropna(subset=["review_scores_rating"])

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=scatter_data,
    x="price",
    y="review_scores_rating",
    alpha=0.5,
    edgecolor=None,
)
plt.title("Listing Price vs Review Score")
plt.xlabel("Nightly price (£)")
plt.ylabel("Review score")
plt.tight_layout()
plt.savefig(figures_dir / "06_price_vs_review_score.png", dpi=150)
plt.show()

scatter_data[["price", "review_scores_rating"]].corr().round(4)

### Business interpretation

If the correlation between price and review score is weak, guests may not be paying
only for reputation. Location, property type, seasonality, and amenities may explain
price better than ratings alone. For hosts, this means a strong review score supports
trust but does not automatically justify a large price premium.

## 7. Listing Density by Neighbourhood

In [ ]:
listing_density = (
    listings.groupby("neighbourhood", as_index=False)
    .agg(listing_count=("listing_id", "count"), median_price=("price", "median"))
    .sort_values("listing_count", ascending=False)
    .head(15)
)

plt.figure(figsize=(12, 6))
sns.barplot(
    data=listing_density,
    x="listing_count",
    y="neighbourhood",
    hue="neighbourhood",
    palette="mako",
    legend=False,
)
plt.title("Top 15 Neighbourhoods by Listing Count")
plt.xlabel("Number of listings")
plt.ylabel("Neighbourhood")
plt.tight_layout()
plt.savefig(figures_dir / "07_listing_density_by_neighbourhood.png", dpi=150)
plt.show()

listing_density

### Business interpretation

Neighbourhoods with the highest listing counts represent the most competitive supply
pools in Edinburgh. These areas may still be attractive to guests, but hosts face more
direct substitutes nearby. Market entrants should expect tighter pricing discipline and
stronger need for positioning, photography, and guest experience in these dense areas.

## Section 04 Summary

This notebook completed the core EDA requirements for Edinburgh:

- Price patterns by neighbourhood and room type
- Host supply concentration
- Review score distribution
- Occupancy proxy comparison
- Price vs review score relationship
- Listing density by neighbourhood

Figures were exported to `reports/figures/` for use in the final report.